# Notebook 3 — ML Models and SHAP Interpretability

Trains XGBoost, Random Forest, and SVM classifiers to predict urban heat
hotspots from landscape predictors, then interprets the models using SHAP.

**Reference:** Hoang N-D et al. (2025). An interpretable machine learning
framework for mapping hotspots and identifying their driving factors in urban
environments during heat waves. *Environ Monit Assess*, 197, 1017.

**Prerequisites:**
- Processed raster stack: `data/processed/ouaga_aligned_stack.tif`
  (produced by `01_processing_pipeline.ipynb`)
- `config/processing.yaml`

**Outputs (saved to `figures/pub/` and `models/`):**
- `shap_bar_importance.png` — global feature importance (bar)
- `shap_beeswarm.png` — direction of influence (beeswarm)
- `shap_dependence_grid.png` — feature value vs SHAP value
- `shap_probability_beeswarm.png` — SHAP in probability space
- `susceptibility_maps.png` — XGBoost / RF / SVM side-by-side
- `binary_hotspot_map.png` — XGBoost binary classification
- `ouaga_predicted_hotspots.tif` — binary hotspot raster
- `ouaga_hotspot_probability.tif` — hotspot probability raster
- `model_comparison_results.csv` — train and test metrics for all models

**Reproduction modes (`MODEL_MODE`):**
- `MODEL_MODE = "load"` (default) reads the frozen pickled models in
  `models/` (archived on Zenodo) and reproduces the exact published
  numbers. **Use this to verify results.**
- `MODEL_MODE = "manual"` retrains with the published best
  hyperparameters (~2 min). Deterministic given fixed `random_state=42`,
  but may differ from the pickled models due to library-version drift.
- `MODEL_MODE = "grid_search"` redoes the full CV tuning (hours).

**Sampling (`USE_SAMPLING`):**
- `False` *(default)* — use all valid pixels.
- `True` — random sample of 4,000 pixels, replicating the reference paper.

---
## 1 · Setup

In [ ]:
from pathlib import Path

import joblib
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import rasterio
import shap
import xgboost as xgb
from matplotlib.patches import Patch
from rasterio.warp import transform_bounds
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

from src.data import load_dataset

# -- Paths ------------------------------------------------------------------
PROJECT_ROOT = Path("..").resolve()
FIG_DIR      = PROJECT_ROOT / "figures" / "pub"
MODEL_DIR    = PROJECT_ROOT / "models"
FIG_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

# -- Reproduction mode ------------------------------------------------------
MODEL_MODE   = "load"   # "load" | "manual" | "grid_search"
USE_SAMPLING = False    # False = all valid pixels | True = 4,000-pixel sample (paper)

# -- Best hyperparameters (copy from grid search output) --------------------
BEST_PARAMS_XGB = {
    "n_estimators": 500,
    "max_depth":     9,
    "learning_rate": 0.2,
    "reg_alpha":     0,
    "reg_lambda":    0.001,
}
BEST_PARAMS_RF = {
    "n_estimators": 300,
    "max_depth":    None,
}
BEST_PARAMS_SVM = {
    "C":      100,
    "gamma":  "scale",
    "kernel": "rbf",
}

---
## 2 · Load Data

In [ ]:
df, config = load_dataset("../config/processing.yaml")

# All bands except the last two (LST, hotspot) are predictors
PREDICTOR_COLS = config["band_names"][:-2]

X = df[PREDICTOR_COLS].values
y = df["hotspot"].values

print(f"Dataset     : {X.shape[0]:,} valid pixels, {X.shape[1]} predictors")
print(f"Predictors  : {PREDICTOR_COLS}")
print(f"Class balance: {np.sum(y == 0):,} non-hotspot ({100*np.mean(y == 0):.1f}%),"
      f" {np.sum(y == 1):,} hotspot ({100*np.mean(y == 1):.1f}%)")

---
## 3 · Data Quality Checks

In [ ]:
print("--- Band descriptions ---")
for i, name in enumerate(config["band_names"], 1):
    print(f"  Band {i}: {name}")

print("\n--- NaN diagnostics ---")
print(f"  Total pixels : {len(y):,}")
print(f"  NaN labels   : {np.isnan(y).sum():,}")
print(f"  Valid labels : {(~np.isnan(y)).sum():,}")

print("\n--- NaNs per predictor band ---")
for i, col in enumerate(PREDICTOR_COLS):
    n_nan = np.isnan(X[:, i]).sum()
    print(f"  {col:25s}: {n_nan:,} NaNs ({100 * n_nan / len(y):.2f}%)")

---
## 4 · Optional Pixel Sampling

In [ ]:
if USE_SAMPLING:
    np.random.seed(42)
    n_samples = min(4_000, X.shape[0])
    idx       = np.random.choice(X.shape[0], size=n_samples, replace=False)
    X_ml, y_ml = X[idx], y[idx]
    print(f"Using {n_samples:,} randomly sampled pixels (paper replication mode)")
else:
    X_ml, y_ml = X, y
    print(f"Using all {X_ml.shape[0]:,} valid pixels")

print(f"\nClass distribution:")
print(f"  Non-hotspot (0): {np.sum(y_ml == 0):,} ({100*np.mean(y_ml == 0):.1f}%)")
print(f"  Hotspot     (1): {np.sum(y_ml == 1):,} ({100*np.mean(y_ml == 1):.1f}%)")

---
## 5 · Train / Test Split

In [ ]:
# 70 / 30 split
X_train, X_test, y_train, y_test = train_test_split(
    X_ml, y_ml,
    test_size=0.3,
    random_state=42,
    # stratify=y_ml  # uncomment to enforce equal class ratios in both subsets
)

print(f"Training : {X_train.shape[0]:,} samples")
print(f"Testing  : {X_test.shape[0]:,} samples")

---
## 6 · Model Training

In [ ]:
if MODEL_MODE == "grid_search":
    print("Running grid search — this will take a while...\n")

    print("Class distribution in training set:")
    dist = np.bincount(y_train.astype(int))
    print(f"  Non-hotspot (0): {dist[0]:,}")
    print(f"  Hotspot     (1): {dist[1]:,}")
    print(f"  Ratio: {dist[0]/dist[1]:.2f}:1\n")

    param_grid_xgb = {
        "n_estimators":  [200, 300, 500],
        "max_depth":     [2, 5, 7, 9],
        "learning_rate": [0.05, 0.1, 0.2],
        "reg_alpha":     [0, 0.00001, 0.0001, 0.001],
        "reg_lambda":    [0, 0.001, 0.01],
    }
    grid_xgb = GridSearchCV(
        xgb.XGBClassifier(random_state=42, eval_metric="logloss"),
        param_grid_xgb, cv=5, scoring="accuracy", n_jobs=-1, verbose=1,
    )
    grid_xgb.fit(X_train, y_train)
    xgb_model = grid_xgb.best_estimator_
    print(f"XGBoost best params: {grid_xgb.best_params_}")

    param_grid_rf = {
        "n_estimators": [50, 100, 200, 300],
        "max_depth":    [None, 5, 10],
    }
    grid_rf = GridSearchCV(
        RandomForestClassifier(random_state=42, n_jobs=-1),
        param_grid_rf, cv=5, scoring="accuracy", n_jobs=-1, verbose=1,
    )
    grid_rf.fit(X_train, y_train)
    rf_model = grid_rf.best_estimator_
    print(f"RF best params: {grid_rf.best_params_}")

    svm_pipe = Pipeline([("scaler", StandardScaler()), ("svc", SVC(probability=True, random_state=42))])
    param_grid_svm = {
        "svc__C":      [1, 10, 100],
        "svc__kernel": ["rbf"],
        "svc__gamma":  [0.01, 0.05, 0.1, "scale"],
    }
    grid_svm = GridSearchCV(svm_pipe, param_grid_svm, cv=5, scoring="accuracy", n_jobs=-1)
    grid_svm.fit(X_train, y_train)
    svm_model = grid_svm.best_estimator_
    print(f"SVM best params: {grid_svm.best_params_}")

    joblib.dump(xgb_model, MODEL_DIR / "xgb_model.pkl")
    joblib.dump(rf_model,  MODEL_DIR / "rf_model.pkl")
    joblib.dump(svm_model, MODEL_DIR / "svm_model.pkl")
    print("Models saved to models/")

elif MODEL_MODE == "manual":
    print("Fitting with known best parameters...\n")

    xgb_model = xgb.XGBClassifier(**BEST_PARAMS_XGB, random_state=42, eval_metric="logloss")
    xgb_model.fit(X_train, y_train)

    rf_model = RandomForestClassifier(**BEST_PARAMS_RF, random_state=42, n_jobs=-1)
    rf_model.fit(X_train, y_train)

    svm_model = Pipeline([
        ("scaler", StandardScaler()),
        ("svc",    SVC(**BEST_PARAMS_SVM, probability=True, random_state=42)),
    ])
    svm_model.fit(X_train, y_train)

    joblib.dump(xgb_model, MODEL_DIR / "xgb_model.pkl")
    joblib.dump(rf_model,  MODEL_DIR / "rf_model.pkl")
    joblib.dump(svm_model, MODEL_DIR / "svm_model.pkl")
    print("Models trained and saved to models/")

elif MODEL_MODE == "load":
    print("Loading saved models from models/...")
    xgb_model = joblib.load(MODEL_DIR / "xgb_model.pkl")
    rf_model  = joblib.load(MODEL_DIR / "rf_model.pkl")
    svm_model = joblib.load(MODEL_DIR / "svm_model.pkl")
    print("Models loaded.")

else:
    raise ValueError(f"Unknown MODEL_MODE: '{MODEL_MODE}'")

---
## 7 · Model Evaluation

Both training and test metrics are reported. 

In [ ]:
def evaluate_model(model, X_train, y_train, X_test, y_test, model_name):
    """Evaluate a classifier on both training and test sets."""

    y_train_pred = model.predict(X_train)
    y_test_pred  = model.predict(X_test)

    def metrics(y_true, y_pred):
        return {
            "Accuracy":  accuracy_score(y_true, y_pred),
            "Precision": precision_score(y_true, y_pred, zero_division=0),
            "Recall":    recall_score(y_true, y_pred, zero_division=0),
            "F1":        f1_score(y_true, y_pred, zero_division=0),
            "Kappa":     cohen_kappa_score(y_true, y_pred),
        }

    tr = metrics(y_train, y_train_pred)
    te = metrics(y_test,  y_test_pred)

    print(f"\n--- {model_name} ---")
    print(f"{'Metric':<12} {'Train':>8} {'Test':>8}")
    print("-" * 30)
    for k in tr:
        print(f"{k:<12} {tr[k]:>8.3f} {te[k]:>8.3f}")

    return {
        "Model":           model_name,
        "Train_Accuracy":  tr["Accuracy"],
        "Train_Precision": tr["Precision"],
        "Train_Recall":    tr["Recall"],
        "Train_F1":        tr["F1"],
        "Train_Kappa":     tr["Kappa"],
        "Test_Accuracy":   te["Accuracy"],
        "Test_Precision":  te["Precision"],
        "Test_Recall":     te["Recall"],
        "Test_F1":         te["F1"],
        "Test_Kappa":      te["Kappa"],
    }


results = [
    evaluate_model(xgb_model, X_train, y_train, X_test, y_test, "XGBoost"),
    evaluate_model(rf_model,  X_train, y_train, X_test, y_test, "Random Forest"),
    evaluate_model(svm_model, X_train, y_train, X_test, y_test, "SVM"),
]

results_df = pd.DataFrame(results)
results_df.to_csv(PROJECT_ROOT / "model_comparison_results.csv", index=False)
print("\nSaved: model_comparison_results.csv")


---
## 8 · Full-Map Prediction and GeoTIFF Export

In [ ]:
data_3d      = config["raster_info"]["data_3d"]
n_predictors = len(PREDICTOR_COLS)
bands, rows, cols = data_3d.shape

# Build full-raster feature matrix (predictors only)
X_full     = data_3d[:n_predictors].reshape(n_predictors, rows * cols).T
valid_mask = ~np.isnan(X_full).any(axis=1)

print(f"Full raster  : {rows} x {cols} = {rows*cols:,} pixels")
print(f"Valid pixels : {valid_mask.sum():,} ({100*valid_mask.mean():.1f}%)")

# Binary prediction
pred_full = np.full(rows * cols, np.nan)
pred_full[valid_mask] = xgb_model.predict(X_full[valid_mask])
pred_map  = pred_full.reshape(rows, cols)

# Probability map
prob_full = np.full(rows * cols, np.nan)
prob_full[valid_mask] = xgb_model.predict_proba(X_full[valid_mask])[:, 1]
prob_map  = prob_full.reshape(rows, cols)
print(f"Probability range: {np.nanmin(prob_map):.3f} to {np.nanmax(prob_map):.3f}")

# Save GeoTIFFs
meta = config["raster_info"]["meta"].copy()
meta.update({"count": 1, "dtype": "float32"})

out_dir      = PROJECT_ROOT / "data" / "processed"
hotspot_path = out_dir / "ouaga_predicted_hotspots.tif"
prob_path    = out_dir / "ouaga_hotspot_probability.tif"

with rasterio.open(hotspot_path, "w", **meta) as dst:
    dst.write(pred_map.astype("float32"), 1)
with rasterio.open(prob_path, "w", **meta) as dst:
    dst.write(prob_map.astype("float32"), 1)

print(f"Saved: {hotspot_path}")
print(f"Saved: {prob_path}")

---
## 9 · SHAP Interpretability

SHAP values decompose each prediction into per-feature contributions.
`TreeExplainer` is exact and fast for tree-based models.
All analyses use the held-out test set.

In [ ]:
explainer   = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

print(f"SHAP values shape     : {shap_values.shape}")
print(f"Base value (log-odds) : {explainer.expected_value:.4f}")

### 9.1 · Global Feature Importance (Bar Chart)

In [ ]:
shap.summary_plot(
    shap_values, X_test,
    feature_names=PREDICTOR_COLS,
    plot_type="bar",
    show=False,
)
plt.gcf().set_size_inches(8, 5)
plt.tight_layout()
plt.savefig(FIG_DIR / "shap_bar_importance.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {FIG_DIR / 'shap_bar_importance.png'}")

# Ranked importance table
global_shap = pd.DataFrame({
    "feature":       PREDICTOR_COLS,
    "mean_abs_shap": np.abs(shap_values).mean(axis=0),
}).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)

print("\nGlobal feature importance (mean |SHAP|):")
print(global_shap.to_string(index=False))

### 9.2 · Direction of Influence (Beeswarm)

In [ ]:
shap.summary_plot(
    shap_values, X_test,
    feature_names=PREDICTOR_COLS,
    show=False,
)
plt.gcf().set_size_inches(8, 5)
plt.tight_layout()
plt.savefig(FIG_DIR / "shap_beeswarm.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {FIG_DIR / 'shap_beeswarm.png'}")

### 9.3 · Dependence Scatter Plots

Feature value vs SHAP value, ordered by descending mean |SHAP| importance.

In [ ]:
# Order features by importance 
features_ordered = global_shap["feature"].tolist()

fig, axes = plt.subplots(3, 3, figsize=(10, 8), dpi=300)
axes = axes.flatten()

feature_cols_list = list(PREDICTOR_COLS)

for idx, feature in enumerate(features_ordered):
    feat_idx = feature_cols_list.index(feature)
    ax = axes[idx]
    ax.scatter(
        X_test[:, feat_idx], shap_values[:, feat_idx],
        alpha=0.5, s=8, c="steelblue", edgecolors="none",
    )
    ax.axhline(y=0, color="red", linestyle="--", linewidth=0.5, alpha=0.7)
    ax.set_xlabel(feature, fontsize=9)
    ax.set_ylabel("SHAP value", fontsize=9)
    ax.tick_params(labelsize=8)
    ax.grid(True, alpha=0.2, linewidth=0.5)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.spines["left"].set_linewidth(0.5)
    ax.spines["bottom"].set_linewidth(0.5)
    ax.text(0.02, 0.98, f"({chr(97 + idx)})",
            transform=ax.transAxes, fontsize=10, fontweight="bold",
            verticalalignment="top")

# Hide unused 9th panel
axes[8].axis("off")

plt.tight_layout(pad=0.5)
plt.savefig(FIG_DIR / "shap_dependence_grid.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print(f"Saved: {FIG_DIR / 'shap_dependence_grid.png'}")

### 9.4 · SHAP in Probability Space

The default `TreeExplainer` operates in log-odds space. Re-computing in
probability space makes contributions directly interpretable as changes
in predicted hotspot probability.

In [ ]:
background_data  = X_train[:100]
explainer_prob   = shap.TreeExplainer(
    xgb_model,
    data=background_data,
    model_output="probability",
    feature_perturbation="interventional",
)
shap_values_prob = explainer_prob.shap_values(X_test)

print(f"Margin base value (log-odds) : {explainer.expected_value:.4f}")
print(f"Probability base value       : {explainer_prob.expected_value:.4f}  "
      f"(avg. likelihood of a hotspot)")

shap.summary_plot(
    shap_values_prob, X_test,
    feature_names=PREDICTOR_COLS,
    show=False,
)
plt.gcf().set_size_inches(8, 5)
plt.tight_layout()
plt.savefig(FIG_DIR / "shap_probability_beeswarm.png", dpi=300, bbox_inches="tight")
plt.show()
print(f"Saved: {FIG_DIR / 'shap_probability_beeswarm.png'}")

---
## 10 · Susceptibility Maps

In [ ]:
# Probability maps for RF and SVM
prob_xgb_flat = np.full(rows * cols, np.nan)
prob_rf_flat  = np.full(rows * cols, np.nan)
prob_svm_flat = np.full(rows * cols, np.nan)

prob_xgb_flat[valid_mask] = xgb_model.predict_proba(X_full[valid_mask])[:, 1]
prob_rf_flat[valid_mask]  = rf_model.predict_proba(X_full[valid_mask])[:, 1]
prob_svm_flat[valid_mask] = svm_model.predict_proba(X_full[valid_mask])[:, 1]

nan_mask = np.isnan(data_3d[0])   # study-area boundary mask

def apply_nan_mask(flat_arr, mask):
    arr = flat_arr.reshape(rows, cols).copy()
    arr[mask] = np.nan
    return arr

prob_xgb_map = apply_nan_mask(prob_xgb_flat, nan_mask)
prob_rf_map  = apply_nan_mask(prob_rf_flat,  nan_mask)
prob_svm_map = apply_nan_mask(prob_svm_flat, nan_mask)

# Lat/lon extent for axis labels
bounds = config["raster_info"]["bounds"]
crs    = config["raster_info"]["crs"]
lon_min, lat_min, lon_max, lat_max = transform_bounds(
    crs, "EPSG:4326", bounds.left, bounds.bottom, bounds.right, bounds.top
)
extent_latlon = [lon_min, lon_max, lat_min, lat_max]

cmap_prob = plt.cm.RdYlGn_r.copy()
cmap_prob.set_bad(color="lightgrey")

fig, axes = plt.subplots(1, 3, figsize=(20, 7), dpi=300)
maps   = [prob_xgb_map, prob_rf_map, prob_svm_map]
titles = ["(a) XGBoost", "(b) Random Forest", "(c) SVM"]

for ax, data, title in zip(axes, maps, titles):
    im = ax.imshow(
        data, cmap=cmap_prob, vmin=0, vmax=1,
        interpolation="none", extent=extent_latlon, origin="upper",
    )
    ax.set_title(title, fontsize=15, fontweight="bold", pad=12)
    ax.set_xlabel("Longitude (\u00b0E)", fontsize=12)
    ax.set_ylabel("Latitude (\u00b0N)", fontsize=12)
    ax.xaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f\u00b0"))
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f\u00b0"))

cbar = fig.colorbar(im, ax=axes, orientation="vertical",
                    fraction=0.03, pad=0.08, shrink=0.75)
cbar.set_label("Hotspot Susceptibility Probability", fontsize=12, labelpad=10)
cbar.set_ticks([0, 0.25, 0.5, 0.75, 1.0])
cbar.set_ticklabels(["0.0\n(Non-hotspot)", "0.25", "0.50", "0.75", "1.0\n(Hotspot)"],
                    fontsize=12)

fig.suptitle("Mapping of Hotspot Susceptibility \u2013 Ouagadougou",
             fontsize=16, fontweight="bold", y=0.96)
plt.tight_layout(rect=[0, 0, 0.88, 1])
plt.savefig(FIG_DIR / "susceptibility_maps.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print(f"Saved: {FIG_DIR / 'susceptibility_maps.png'}")

## 11 · Binary Hotspot Map (XGBoost)

In [ ]:
pred_binary_masked = pred_map.copy()
pred_binary_masked[nan_mask] = np.nan

cmap_binary = mcolors.ListedColormap(["#27ae60", "#c0392b"])
norm        = mcolors.BoundaryNorm([-0.5, 0.5, 1.5], cmap_binary.N)
cmap_binary.set_bad(color="#d5d8dc")

fig, ax = plt.subplots(figsize=(9, 8), dpi=300)
ax.imshow(pred_binary_masked, cmap=cmap_binary, norm=norm, interpolation="none")
ax.set_title("XGBoost \u2013 Binary Hotspot Classification\nOuagadougou",
             fontsize=14, fontweight="bold", pad=12)
ax.axis("off")

legend_elements = [
    Patch(facecolor="#27ae60", edgecolor="grey", label="Non-hotspot"),
    Patch(facecolor="#c0392b", edgecolor="grey", label="Hotspot"),
    Patch(facecolor="#d5d8dc", edgecolor="grey", label="No data"),
]
ax.legend(handles=legend_elements, loc="upper left", fontsize=11,
          framealpha=0.9, edgecolor="grey")

plt.tight_layout()
plt.savefig(FIG_DIR / "binary_hotspot_map.png", dpi=300, bbox_inches="tight", facecolor="white")
plt.show()
print(f"Saved: {FIG_DIR / 'binary_hotspot_map.png'}")